# Tutorial 08 — Learning a 1D linear PMM from real data

This notebook walks through the parameter-learning workflow shipped in `prg/learning/`:

1. Load a real two-column time series (wind-farm active power and wind speed).
2. Estimate the five PMM parameters `(a, b, c, d, e)` by the method of moments.
3. Compare the fitted PMM to its HMM projection `(c = a·b², d = e = a·b)` — when the gap is large, the data is not well described by a classical Kalman model and the **pairwise** parametrisation matters.
4. Convert the parameters to the `LinearAmQ` form (`A`, `B`, `mQ`, `mz0`, `Pz0`) ready to feed into the linear PKF.

**Scope.** The estimator is limited to the **scalar** linear case (`dim_x = dim_y = 1`). For nonlinear EPKF/UPKF models, parameter identification is a separate problem and is **not** covered here.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from prg.learning import estimate_pmm_params, pmm_to_linear_params, validate_pmm

## 1. Load the sample data

We use the WindFarms sample shipped under `data/samples/windfarms/`: 586 hourly readings of active power and wind speed at one onshore site over October 2022, already standardised to zero mean and unit variance per column.

In [ ]:
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
csv_path  = repo_root / 'data' / 'samples' / 'windfarms' / 'site1_202210_Month_586_norm.csv'
df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
print(df.shape)
df.head()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
df['ActivePower_KWh'].plot(ax=ax[0], color='C0', lw=0.8); ax[0].set_ylabel('X — Active power (norm.)')
df['WindSpeed'].plot(ax=ax[1],     color='C1', lw=0.8); ax[1].set_ylabel('Y — Wind speed (norm.)')
ax[1].set_xlabel('time'); plt.tight_layout(); plt.show()

## 2. Estimate the PMM parameters

`estimate_pmm_params` computes the empirical 4×4 covariance of the lagged vector `(X_n, Y_n, X_{n+1}, Y_{n+1})`, then reads off the five entries. Setting `standardise=False` here because the file is already standardised; pass the original-units file instead and the estimator will centre/scale internally.

In [ ]:
params = estimate_pmm_params(df, x_col='ActivePower_KWh', y_col='WindSpeed', standardise=False, verbose=1)
assert validate_pmm(params), 'Estimated parameters are not a valid PMM.'
params

## 3. PMM vs HMM (classical Kalman) projection

The classical Kalman model corresponds to the constrained submanifold `c = a·b²`, `d = e = a·b`. If the data lies far from this submanifold, the unconstrained PMM is a strictly richer description.

In [ ]:
hmm = params.as_hmm()
print(f'PMM : {params}')
print(f'HMM : {hmm}')
print(f'Δc  = {params.c - hmm.c:+.4f}')
print(f'Δd  = {params.d - hmm.d:+.4f}')
print(f'Δe  = {params.e - hmm.e:+.4f}')
print(f'Is the fit exactly HMM ? {params.is_hmm()}')

## 4. Convert to `LinearAmQ` form

`pmm_to_linear_params` returns the kwargs (`A`, `B`, `mQ`, `mz0`, `Pz0`) that feed the linear pairwise model — `dim_x = dim_y = 1`, so the pairwise state is the 2-vector `z = (X, Y)`.

In [ ]:
linear_kw = pmm_to_linear_params(params)
for key, val in linear_kw.items():
    print(f'{key} =\n{np.array2string(val, precision=4)}\n')

## Next steps

From here the kwargs can be plugged into a `LinearAmQ` instance (with `dim_x=1`, `dim_y=1`, `pairwiseModel=True`) and then wrapped in `ParamLinear` to drive `Linear_PKF` — see `notebooks/tutorial_01_getting_started.ipynb` for the linear-filter API.

A natural extension is to compare the **PMM-fitted** filter against the **HMM-projected** one on the same series and report the MSE gain — when `params.is_hmm()` is far from `True`, the PMM is expected to track the state more closely.